# Colab: real-data PINN, batch 4 (final runs), **layout 0**

The final runs are split over 3 notebooks, one per block layout (L0, L1, L2), so 3 people can run them at the same time.
**You are running layout 0: 8 runs, about 3.5 hours.**

Settings are fixed (chosen in batch 3 on validation, by rules set in advance): canal seepage on, zoned design = shared.
Each run trains one of the two compared models: `regional` (one PINN for the whole area) or `shared` (our zoned PINN).
Training scores validation only; the test wells stay sealed.

**Do not change any cell.** Just follow these steps:
1. Put `pinn_code.zip` (from the project lead) in a Google Drive folder called `pinn` (so: `MyDrive/pinn/pinn_code.zip`).
2. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Allow Google Drive access when asked.
3. Keep the laptop awake (plugged in; sleep off; this tab open). If Colab disconnects, just **Run all** again: finished runs are skipped.
4. When the runs are done (or you have to stop), run the **last cell** and send `MyDrive/pinn/real4_L0_results.zip` to the project lead.

In [ ]:
LAYOUTS = [0]
SEEDS = [0, 1, 2, 3, 4]
DRIVE_ROOT = "/content/drive/MyDrive/pinn"

In [ ]:
import glob
import os
import shutil
import zipfile
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/runs/logs", exist_ok=True)
CODE = "/content/pinn"
shutil.rmtree(CODE, ignore_errors=True)
with zipfile.ZipFile(f"{DRIVE_ROOT}/pinn_code.zip") as z:
    z.extractall(CODE)
%cd {CODE}

In [ ]:
# PyTorch is already installed on Colab; this adds everything else (about 1 minute).
!pip install -q -r requirements.txt
!pip install -q -e .
os.environ["PINN_RUNS_DIR"] = f"{DRIVE_ROOT}/runs"
!python scripts/check_env.py
!python -m pytest -q -m "not data"

## Runs

In [ ]:
def done(pattern):
    return bool(glob.glob(f"{DRIVE_ROOT}/runs/{pattern}/metrics.json"))

LOG = f"{DRIVE_ROOT}/runs/logs/real4_L0.log"
MODELS = [("kmeansk5_shared", "--shared"), ("regional", "--regional")]
for seed in SEEDS:
    for layout in LAYOUTS:
        for name, flag in MODELS:
            if layout == 0 and seed == 0:
                continue   # done in batch 3
            if done(f"*_real_L{layout}_{name}_s{seed}_tbc"):
                print("already done:", name, layout, seed)
                continue
            !python -u scripts/run_real.py {flag} --layout {layout} --seed {seed} --tag tbc --set canal_seepage.strength=1 --stage-only val 2>&1 | tee -a {LOG}

## Bundle the results for Claude
Makes `MyDrive/pinn/real4_L0_results.zip` with every **finished** final run (run it any time).
Download it, put it in the `pinn` folder on the laptop, then tell Claude.

In [ ]:
out = f"{DRIVE_ROOT}/real4_L0_results.zip"
n = 0
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for name, _ in MODELS:
        for run in glob.glob(f"{DRIVE_ROOT}/runs/*_real_L0_{name}_s*_tbc"):
            if not os.path.exists(f"{run}/metrics.json"):
                continue
            n += 1
            for path in glob.glob(f"{run}/*"):
                z.write(path, os.path.relpath(path, f"{DRIVE_ROOT}/runs"))
    if os.path.exists(LOG):
        z.write(LOG, "logs/real4_L0.log")
print("wrote", out, f"({n} finished runs, {os.path.getsize(out) / 1e6:.2f} MB)")